# Coculture Treated - Mechanical Automatic Modeling
Decode processed data, fit coculture treated model zoo, and run sensitivity/uncertainty analysis.

In [1]:
import Pkg
Pkg.activate(joinpath(@__DIR__, ".."))
Pkg.instantiate()

  Activating project at `~/Desktop/Research/CancerGrowthDynamics/Modeling_Approaches/02_mechanical_automatic_package`


In [4]:
using CSV, DataFrames, Plots, Dates
include(joinpath(@__DIR__, "..", "src", "MechanicalAutomaticModeling.jl"))
using .MechanicalAutomaticModeling
using GrowthParameterEstimation

In [5]:
condition = "coculture_treated"
decoded = MechanicalAutomaticModeling.IOUtils.decode_condition_dataframe(condition; start=@__DIR__)
out = MechanicalAutomaticModeling.IOUtils.condition_output_dirs(condition; start=@__DIR__)
decoded_path = joinpath(out.csv, "$(condition)_automatic_decoded.csv")
CSV.write(decoded_path, decoded)
MechanicalAutomaticModeling.IOUtils.write_manifest_row(condition=condition, step="decode", outputs=[decoded_path], start=@__DIR__)
first(decoded, min(10, nrow(decoded)))

Row,time,count,source_file,condition,density,cell_line,dose,mix
,Float64,Float64,String,String,String,String,Float64,String
1,1.0,88.59,combined_day_averages.csv,coculture_treated,,,0.0,
2,1.0,51.28,combined_day_averages.csv,coculture_treated,,,0.0,
3,1.0,26.93,combined_day_averages.csv,coculture_treated,,,0.0,
4,1.0,99.62,combined_day_averages.csv,coculture_treated,,,0.0,
5,1.0,154.28,combined_day_averages.csv,coculture_treated,,,0.0,
6,1.0,216.95,combined_day_averages.csv,coculture_treated,,,0.0,
7,1.0,87.41,combined_day_averages.csv,coculture_treated,,,0.0,
8,1.0,53.36,combined_day_averages.csv,coculture_treated,,,0.0,
9,1.0,28.45,combined_day_averages.csv,coculture_treated,,,0.0,


In [6]:
fit_artifacts = MechanicalAutomaticModeling.FitWorkflows.run_condition_fit!(decoded, condition; start=@__DIR__)
first(fit_artifacts.ranking, min(10, nrow(fit_artifacts.ranking)))

Row,model,sse,weighted_sse,aic,bic,n_params,delta_bic
,String,Float64,Float64,Float64,Float64,Int64,Float64
1,theta_logistic_hill_kill,1.0e12,1.0e12,46741.5,46776.1,6,0.0
2,pkpd_inhibition,1.0e12,1.0e12,46743.5,46783.9,7,7.76302
3,lotka_volterra_hill_competition,1.0e12,1.0e12,46751.5,46814.9,11,38.8151


In [ ]:
analysis_artifacts = MechanicalAutomaticModeling.AnalysisWorkflows.run_condition_analysis!(decoded, fit_artifacts, condition; start=@__DIR__)
analysis_artifacts.sensitivity

In [ ]:
out = MechanicalAutomaticModeling.IOUtils.condition_output_dirs(condition; start=@__DIR__)
summary = DataFrame(
    condition = [condition],
    decoded_rows = [nrow(decoded)],
    fit_rows = [nrow(fit_artifacts.ranking)],
    sensitivity_rows = [nrow(analysis_artifacts.sensitivity)],
    generated_at = [Dates.format(now(UTC), dateformat"yyyy-mm-ddTHH:MM:SS")]
)
summary_path = joinpath(out.metrics, "$(condition)_automatic_summary.csv")
CSV.write(summary_path, summary)
MechanicalAutomaticModeling.IOUtils.write_manifest_row(condition=condition, step="summary", outputs=[summary_path], start=@__DIR__)
summary